# Bölüm 9: Gömme katmanları

> "Bir kelimenin ne olduğunu, etrafındaki kelimelerden anlarsınız." — **John Rupert Firth**, Dilbilimci

---

## Neler öğreneceksin

- Token kimliklerinin neden yeterli olmadığı ve gömmelerin gerçekte neyi çözdüğü
- Arama tabloları kullanarak sıfırdan token gömmelerinin nasıl oluşturulacağı
- Konum bilgisinin neden önemli olduğu ve nasıl ekleneceği
- Tam girdi temsilleri için token ve konum gömmelerinin nasıl birleştirileceği
- Gerçek model gömmelerinin nasıl keşfedileceği ve anlamsal ilişkilerin nasıl bulunacağı
- Pratik başlatma ve uygulama hususları

---

## Kurulum

Önce gerekli paketleri yükleyelim:

In [ ]:
# Gerekli paketleri yükle
!pip install -q torch transformers

In [ ]:
# ===== İÇE AKTARMALAR =====
import torch                     # PyTorch: tensör işlemleri ve sinir ağları
import torch.nn as nn            # Sinir ağı modülleri (katmanlar, vb.)
import torch.nn.functional as F  # Matematiksel fonksiyonlar (cosine_similarity, vb.)
from transformers import AutoModel, AutoTokenizer  # Önceden eğitilmiş modeller

# Hızlı tensör hatırlatması:
# - torch.tensor([1,2,3]) 1 boyutlu tensör oluşturur (liste gibi)
# - torch.randn(3, 4) rastgele sayılardan oluşan 3×4 tensör oluşturur
# - tensor[0] ilk boyuta indeksleme yapar
# - tensor.shape boyutları gösterir

## 1. Neden sadece token kimliklerini kullanamayız?

Token kimlikleri yalnızca tam sayı indeksleridir; sayısal yakınlık anlamsal yakınlık göstermez. Sorunu görelim:

In [ ]:
# Token kimlikleri sadece tam sayıdır - ilişki yoktur
# (gerçek GPT-2 kimlikleri; baştaki boşluklar önemli: " cat" ve "cat")
token_ids = {
    " cat": 3797,
    " kitten": 42143,
    " dog": 3290,
    " car": 1097
}

print("Token kimlikleri:")
for word, id in token_ids.items():
    print(f"  '{word}' → {id}")

# Sorun: "cat" sayısal olarak "kitten"den çok "dog"a daha yakın!
print(f"\n'cat' ile 'dog' arası mesafe: {abs(3797 - 3290)}")
print(f"'cat' ile 'kitten' arası mesafe: {abs(3797 - 42143)}")
print("\nAma anlamsal olarak 'cat' ve 'kitten' daha benzer!")

### Çözüm: gömmeler

Her token kimliğini anlamı yakalayan yoğun bir vektöre dönüştür.

**`nn.Embedding` nedir?**
- `vocab_size` satır ve `embed_dim` sütunlu bir arama tablosu
- Her satır bir token'ı temsil eden bir vektördür
- Girdi: token kimliği (tam sayı) → Çıktı: o satır (vektör)
- Bir sözlük gibi düşün: `{0: [0.1, 0.2, ...], 1: [0.5, -0.1, ...], ...}`

In [ ]:
# ===== Sorun: Token kimlikleri =====
token_ids = torch.tensor([3797, 42143, 3290])  # " cat", " kitten", " dog"
print(f"Token kimlikleri şekli: {token_ids.shape}")
print(f"Sadece tam sayılar: {token_ids}")

# ===== Çözüm: Gömmeler =====
vocab_size = 50257  # GPT-2 kelime dağarcığı
embed_dim = 768     # GPT-2 gömme boyutu

# Gömme katmanı oluştur (bu bir arama tablosudur!)
embedding = nn.Embedding(vocab_size, embed_dim)

# Token kimliklerimiz için vektörleri ara
token_vectors = embedding(token_ids)
print(f"\nToken gömmeleri şekli: {token_vectors.shape}")
print(f"İlk token'ın vektörü (ilk 10 boyut): {token_vectors[0, :10]}")

# Artık her token anlam yakalayabilen 768 boyutlu bir vektördür!

## 2. Token gömmeleri: arama tablosu

Mekanizmayı anlamak için sıfırdan token gömmeleri oluşturalım.

### Adım 1: gömme matrisini oluştur

In [ ]:
vocab_size = 50257  # GPT-2 kelime dağarcığı boyutu
embed_dim = 768     # Gömme boyutu

# Gömme matrisi oluştur: her token için bir satır
embedding_matrix = torch.randn(vocab_size, embed_dim)

print(f"Gömme matrisi şekli: {embedding_matrix.shape}")
print(f"\nToken 3797'nin gömmesi (ilk 10 boyut): {embedding_matrix[3797, :10]}")

### Adım 2: birden fazla token'ı ara

In [ ]:
token_ids = torch.tensor([464, 3797, 3332])  # "The cat sat"

# Manuel arama (gömme katmanlarının dahili olarak yaptığı)
embeddings = embedding_matrix[token_ids]

print(f"Girdi şekli: {token_ids.shape}")       # torch.Size([3])
print(f"Çıktı şekli: {embeddings.shape}")     # torch.Size([3, 768])

# Her token kimliği → onun 768 boyutlu vektörü
print(f"\nToken 464'ün gömmesi (ilk 5 boyut): {embeddings[0, :5]}")
print(f"Token 3797'nin gömmesi (ilk 5 boyut): {embeddings[1, :5]}")
print(f"Token 3332'nin gömmesi (ilk 5 boyut): {embeddings[2, :5]}")

### Adım 3: yığınları işle

In [ ]:
# Her biri 4 token'lı 2 diziden oluşan yığın
token_ids_batch = torch.tensor([
    [464, 3797, 3332, 319],    # Dizi 1: "The cat sat on"
    [40, 481, 1037, 345]       # Dizi 2: "I will help you"
])

print(f"Yığın şekli: {token_ids_batch.shape}")  # torch.Size([2, 4])

# Tüm yığın için gömmeleri ara
embeddings_batch = embedding_matrix[token_ids_batch]

print(f"Gömmelerin şekli: {embeddings_batch.shape}")  # torch.Size([2, 4, 768])
print("\nŞekil dönüşümü: (batch, seq) → (batch, seq, embed_dim)")

### Adım 4: PyTorch'un nn.Embedding'ini kullan

In [ ]:
class TokenEmbedding(nn.Module):
    """
    Token gömme katmanı: token kimliklerini yoğun vektörlere dönüştürür.
    
    Bu, GPT-2'nin 'wte' (kelime token gömmeleri) katmanının yaptığı şeydir.
    """
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # Gömme matrisini öğrenilebilir parametre olarak oluştur
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # Küçük rastgele değerlerle başlat (GPT-2 stili)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
    
    def forward(self, token_ids):
        """
        Args:
            token_ids: (batch, seq) token kimlikleri tensörü
        
        Returns:
            embeddings: (batch, seq, embed_dim) token vektörleri tensörü
        """
        return self.embedding(token_ids)

# Token gömme katmanı oluştur
token_embed = TokenEmbedding(vocab_size=50257, embed_dim=768)

# Bir yığın göm
token_ids = torch.tensor([[464, 3797, 3332, 319]])  # Şekil: (1, 4)
embeddings = token_embed(token_ids)

print(f"Girdi şekli: {token_ids.shape}")        # torch.Size([1, 4])
print(f"Çıktı şekli: {embeddings.shape}")      # torch.Size([1, 4, 768])
print(f"\nİlk token gömmesi (ilk 5 boyut): {embeddings[0, 0, :5]}")

## 3. Konum gömmeler: konumu öğretmek

Token gömmelerinin konum algısı yoktur. Sorunu görelim:

In [ ]:
# Aynı token'lara sahip, farklı sıralı iki dizi
token_ids_1 = torch.tensor([[464, 3797, 3332]])  # "The cat sat"
token_ids_2 = torch.tensor([[3332, 3797, 464]])  # "sat cat The"

# Token gömmelerini al
token_embed = TokenEmbedding(vocab_size=50257, embed_dim=768)
embeddings_1 = token_embed(token_ids_1)
embeddings_2 = token_embed(token_ids_2)

print(f"Gömmeler 1 şekli: {embeddings_1.shape}")
print(f"Gömmeler 2 şekli: {embeddings_2.shape}")

# Farklılar...
print(f"\nGömmeler aynı mı? {torch.equal(embeddings_1, embeddings_2)}")

# Vektörlerin içindeki sayıları değil, ikinci dizinin token sırasını tersine çevir.
assert torch.equal(embeddings_1, embeddings_2.flip(1))
print("Aynı token vektörleri ters sırada: arama tablosu konum bilgisi eklemez.")

### Öğrenilen konum gömmeleri uygulama

In [ ]:
class PositionalEmbedding(nn.Module):
    """
    Öğrenilen konum gömmeler: her konum için bir eğitilebilir vektör.
    
    GPT-2 bu yaklaşımı kullanır ('wpe' - kelime konum gömmeleri olarak adlandırılır).
    """
    def __init__(self, max_seq_len, embed_dim):
        """
        Args:
            max_seq_len: Maksimum dizi uzunluğu (örn. GPT-2 için 1024)
            embed_dim: Gömme boyutu (token gömmeleri ile eşleşmeli)
        """
        super().__init__()
        # Konum gömme matrisi oluştur: (max_seq_len, embed_dim)
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        
        # Küçük rastgele değerlerle başlat (GPT-2 stili)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)
        
        self.max_seq_len = max_seq_len
    
    def forward(self, token_ids):
        """
        Args:
            token_ids: (batch, seq) token kimlikleri tensörü
        
        Returns:
            pos_embeddings: (batch, seq, embed_dim) konum vektörleri tensörü
        """
        batch_size, seq_len = token_ids.shape
        
        # Dizi uzunluğunu doğrula
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Dizi uzunluğu {seq_len}, max_seq_len {self.max_seq_len}'i aşıyor"
            )
        
        # Konum indekslerini oluştur: [0, 1, 2, ..., seq_len-1]
        position_ids = torch.arange(
            seq_len,
            device=token_ids.device  # Girdinin cihazını (CPU/GPU) eşleştir
        )
        
        # Yığın için genişlet: (seq_len,) → (batch, seq_len)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        
        # Konum gömmelerini ara
        pos_embeddings = self.pos_embed(position_ids)
        
        return pos_embeddings

# Konum gömme katmanı oluştur
pos_embed = PositionalEmbedding(max_seq_len=1024, embed_dim=768)

# Örnek: 4 uzunluğunda dizi
token_ids = torch.tensor([[464, 3797, 3332, 319]])  # Şekil: (1, 4)
pos_embeddings = pos_embed(token_ids)

print(f"Girdi şekli: {token_ids.shape}")           # torch.Size([1, 4])
print(f"Konum gömmeleri şekli: {pos_embeddings.shape}")  # torch.Size([1, 4, 768])

# Her konum kendi öğrenilen vektörünü alır
print(f"\nKonum 0 gömmesi (ilk 5 boyut): {pos_embeddings[0, 0, :5]}")
print(f"Konum 1 gömmesi (ilk 5 boyut): {pos_embeddings[0, 1, :5]}")
print(f"Konum 2 gömmesi (ilk 5 boyut): {pos_embeddings[0, 2, :5]}")
print(f"Konum 3 gömmesi (ilk 5 boyut): {pos_embeddings[0, 3, :5]}")

## 4. Token + konum gömmelerini birleştirme

### Tam GPT2Embeddings sınıfını oluşturma

In [ ]:
class GPT2Embeddings(nn.Module):
    """
    Tam GPT-2 gömme katmanı: token + konum gömmeleri.
    
    Bu, GPT-2'nin 'wte' + 'wpe' katmanlarıyla eşleşir.
    """
    def __init__(self, vocab_size, max_seq_len, embed_dim):
        """
        Args:
            vocab_size: Kelime dağarcığı boyutu (GPT-2 için 50257)
            max_seq_len: Maksimum dizi uzunluğu (GPT-2 için 1024)
            embed_dim: Gömme boyutu (GPT-2 Small için 768)
        """
        super().__init__()
        
        # Token gömmeleri: vocab_size × embed_dim
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        
        # Konum gömmeleri: max_seq_len × embed_dim
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        
        # Her ikisini de GPT-2'nin standart başlatması ile başlat
        nn.init.normal_(self.token_embed.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)
        
        self.max_seq_len = max_seq_len
    
    def forward(self, token_ids):
        """
        Args:
            token_ids: (batch, seq) token kimlikleri tensörü
        
        Returns:
            embeddings: (batch, seq, embed_dim) birleştirilmiş gömmeler tensörü
        """
        batch_size, seq_len = token_ids.shape
        
        # Dizi uzunluğunu doğrula
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Dizi uzunluğu {seq_len}, max_seq_len {self.max_seq_len}'i aşıyor"
            )
        
        # ===== Token Gömmeleri =====
        token_embeddings = self.token_embed(token_ids)
        
        # ===== Konum Gömmeleri =====
        position_ids = torch.arange(seq_len, device=token_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        position_embeddings = self.pos_embed(position_ids)
        
        # ===== Toplama ile Birleştir =====
        embeddings = token_embeddings + position_embeddings
        
        return embeddings

# GPT-2 Small gömme katmanı oluştur
gpt2_embed = GPT2Embeddings(
    vocab_size=50257,
    max_seq_len=1024,
    embed_dim=768
)

# Örnek: bir yığın dizi göm (HER İKİSİ de aynı uzunlukta olmalı!)
# GPT-2'nin özel bir dolgu token'ı yoktur: 0 kimliği '!'dir, dolgu değil.
# Yaygın yaklaşım (Bölüm 12), metin sonu token'ı olan 50256 ile doldurmaktır.
token_ids = torch.tensor([
    [464, 3797, 3332, 319, 50256, 50256],  # "The cat sat on" + dolgu
    [40, 481, 1037, 345, 50256, 50256]     # "I will help you" + dolgu
])

embeddings = gpt2_embed(token_ids)

print(f"Girdi şekli: {token_ids.shape}")         # torch.Size([2, 6])
print(f"Çıktı şekli: {embeddings.shape}")       # torch.Size([2, 6, 768])
print(f"Her token artık şu boyuta sahip: {embeddings.shape[-1]} boyut")

print(f"\nİlk dizi, ilk token (ilk 10 boyut):")
print(embeddings[0, 0, :10])

### Bölüm 8'le bağlantı: tam boru hattı

In [ ]:
from transformers import AutoTokenizer

# ===== Adım 1: Tokenize Et (Bölüm 8) =====
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "The cat sat on the mat"
token_ids = tokenizer.encode(text, return_tensors="pt")

print("Adım 1: Tokenizasyon")
print(f"Metin: {text}")
print(f"Token kimlikleri: {token_ids}")
print(f"Şekil: {token_ids.shape}\n")

# ===== Adım 2: Göm (Bölüm 9) =====
gpt2_embed = GPT2Embeddings(vocab_size=50257, max_seq_len=1024, embed_dim=768)
embeddings = gpt2_embed(token_ids)

print("Adım 2: Gömme")
print(f"Gömmelerin şekli: {embeddings.shape}")
print(f"İlk token gömmesi (ilk 10 boyut): {embeddings[0, 0, :10]}\n")

print("Adım 3: Sırada — Dikkat katmanları (Bölüm 10)")
print(f"Bu {embeddings.shape} gömmeler öz-dikkate akacak,")
print("burada token'lar birbirlerinin bağlamından öğrenecek!")

## 5. GPT-2'nin gömmelerini keşfetme

Gerçek bir önceden eğitilmiş GPT-2 modeli yükle ve ne öğrendiğini keşfet:

In [ ]:
# GPT-2 Small'ı yükle
model = AutoModel.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Token gömmelerine eriş
token_embeddings = model.wte.weight  # Kelime Token Gömmeleri
print(f"Token gömmeleri şekli: {token_embeddings.shape}")
# torch.Size([50257, 768]) — her token için bir 768 boyutlu vektör

# Konum gömmelerine eriş
position_embeddings = model.wpe.weight  # Kelime Konum Gömmeleri
print(f"Konum gömmeleri şekli: {position_embeddings.shape}")
# torch.Size([1024, 768]) — her konum için bir 768 boyutlu vektör

### Benzer kelimeleri bulma

**Kosinüs Benzerliği nedir?**
İki vektörün aralarındaki açıya göre ne kadar benzer olduğunu ölçer:
- **1.0** = aynı yön (çok benzer)
- **0.0** = dik (ilgisiz)
- **-1.0** = zıt yön (zıt anlam)

Bunu şöyle düşün: "bu iki ok aynı yöne mi işaret ediyor?"

In [ ]:
def find_similar_tokens(word, embeddings, tokenizer, top_k=5):
    """Verilen kelimeye en benzer gömmelere sahip token'ları bul."""
    # Kelime için token kimliğini al
    token_id = tokenizer.encode(word, add_special_tokens=False)[0]
    target_vec = embeddings[token_id]
    
    # Tüm token'larla kosinüs benzerliğini hesapla
    similarities = F.cosine_similarity(
        target_vec.unsqueeze(0),  # (1, 768)
        embeddings,               # (50257, 768)
        dim=1
    )
    
    # En benzer top-k'yı al (kelimenin kendisini hariç tut)
    top_indices = similarities.argsort(descending=True)[1:top_k+1]
    
    print(f"\n'{word}' kelimesine en benzer kelimeler:")
    for idx in top_indices:
        token = tokenizer.decode([idx])
        score = similarities[idx].item()
        print(f"  {score:.3f} — '{token}'")

# Anlamsal ilişkileri keşfet
find_similar_tokens("king", token_embeddings, tokenizer, top_k=8)
find_similar_tokens("computer", token_embeddings, tokenizer, top_k=8)
find_similar_tokens("happy", token_embeddings, tokenizer, top_k=8)

### Kendi keşiflerini dene!

In [ ]:
# Farklı kelimeleri dene:
words_to_explore = ["Python", "doctor", "fast", "beautiful"]

for word in words_to_explore:
    find_similar_tokens(word, token_embeddings, tokenizer, top_k=5)

### Bonus: kelime analojileri

"king - man + woman ≈ queen" denklemini sağlayan vektörleri bulabilir miyiz?

In [ ]:
def word_analogy(word1, word2, word3, embeddings, tokenizer, top_k=5):
    """
    Bul: word1 - word2 + word3 ≈ ?
    Örnek: king - man + woman ≈ queen
    """
    # Token kimliklerini al
    id1 = tokenizer.encode(word1, add_special_tokens=False)[0]
    id2 = tokenizer.encode(word2, add_special_tokens=False)[0]
    id3 = tokenizer.encode(word3, add_special_tokens=False)[0]
    
    # Hedef vektörü hesapla: word1 - word2 + word3
    target_vec = embeddings[id1] - embeddings[id2] + embeddings[id3]
    
    # En benzer token'ları bul
    similarities = F.cosine_similarity(
        target_vec.unsqueeze(0),
        embeddings,
        dim=1
    )
    
    # Girdi kelimelerini sonuçlardan hariç tut
    similarities[id1] = -1
    similarities[id2] = -1
    similarities[id3] = -1
    
    top_indices = similarities.argsort(descending=True)[:top_k]
    
    print(f"\n{word1} - {word2} + {word3} ≈ ?")
    for idx in top_indices:
        token = tokenizer.decode([idx])
        score = similarities[idx].item()
        print(f"  {score:.3f} — '{token}'")

# Klasik örnek: king - man + woman ≈ queen
word_analogy("king", "man", "woman", token_embeddings, tokenizer, top_k=5)

# Diğerlerini dene!
word_analogy("Paris", "France", "Germany", token_embeddings, tokenizer, top_k=5)

## 6. Uygulamalı alıştırmalar

### Alıştırma 1: manuel gömme araması

```python
# Küçük bir kelime dağarcığı (10 token) ve 8 gömme boyutu oluştur
# Manuel olarak bir gömme matrisi oluştur ve [2, 5, 7] token kimlikleri için gömmeleri ara
# Her adımda şekilleri yazdır

# Adım 1: Gömme matrisi oluştur
vocab_size = 10
embed_dim = 8
embedding_matrix = torch.randn(???, ???)  # Boyutları doldur
print(f"Gömme matrisi şekli: {embedding_matrix.shape}")

# Adım 2: Aranacak token kimliklerini oluştur
token_ids = torch.tensor([2, 5, 7])
print(f"Token kimlikleri: {token_ids}")

# Adım 3: Gömmeleri ara (ipucu: embedding_matrix[...] gibi indeksleme kullan)
embeddings = ???
print(f"Gömmelerin şekli: {embeddings.shape}")
```

### Alıştırma 2: TokenEmbedding'i sıfırdan oluştur

In [ ]:
# Örneğe bakmadan TokenEmbedding sınıfını uygula
# Uygun başlatma ve şekil doğrulaması ekle

# KODUNU BURAYA YAZ

### Alıştırma 3: konum kodlaması

In [ ]:
# Token kimlikleri dizisi oluştur: [10, 20, 30, 40, 50]
# Konum indeksleri oluştur ve bunları bir konum gömme katmanında ara
# Konum 0'ın token'dan bağımsız olarak her zaman aynı vektörü aldığını doğrula

# KODUNU BURAYA YAZ

### Alıştırma 4: tam GPT2Embeddings

In [ ]:
# Tam GPT2Embeddings sınıfını uygula
# Şununla test et:
# - 2 diziden oluşan bir yığın
# - Farklı dizi uzunlukları (biri 5 uzunluğunda, diğeri dolgu ile 8)
# - Çıktı şeklinin (2, 8, embed_dim) olduğunu doğrula

# KODUNU BURAYA YAZ

### Alıştırma 5: GPT-2 benzerliklerini keşfet

In [ ]:
# GPT-2'yi yükle ve şunlara benzer token'ları bul:
# - "Python" (programlama ile ilgili kelimeler bulmalı)
# - "doctor" (tıbbi/profesyonel kelimeler bulmalı)
# - "fast" (hızla ilgili kelimeler bulmalı)
#
# Model ilgili kavramları birlikte gruplayabiliyor mu?

# KODUNU BURAYA YAZ

### Alıştırma 6: cihaz uyuşmazlığı

In [ ]:
# Bir GPT2Embeddings örneği oluştur
# GPU'ya taşı (varsa)
# CPU'da başlayan token kimliklerini gömmek için kullan
# Ne olur? Token kimliklerini önce GPU'ya taşıyarak düzelt

# KODUNU BURAYA YAZ

### Alıştırma 7: dizi uzunluğu sınırları

In [ ]:
# max_seq_len=10 olan bir GPT2Embeddings oluştur
# 15 uzunluğunda bir diziyi gömmeye çalış
# Hatayı, gömmeden önce diziyi keserek nazikçe ele al

# KODUNU BURAYA YAZ

### Alıştırma 8: Bölüm 8 entegrasyonu

In [ ]:
# Bölüm 8'den JSONL çıktısını al (tokenize edilmiş veri kümen)
# Bir kayıt yükle, token kimliklerini çıkar, PyTorch tensörüne dönüştür
# GPT2Embeddings kullanarak gömmek için kullan
# Her adımda şekli yazdır

# KODUNU BURAYA YAZ

## Bölüm özeti

**Ne oluşturduk:**

1. **Token gömmeleri:** Token kimliklerini anlamsal vektörlere dönüştüren arama tablosu
2. **Konum gömmeler:** Dizideki konumu kodlayan öğrenilmiş vektörler
3. **Tam GPT2Embeddings:** Token + konumu toplama yoluyla birleştirir

**Ne öğrendik:**

- Token kimlikleri anlamsal anlamı olmayan rastgele indekslerdir
- Gömmeler, kimlikleri ilişkileri yakalayan yoğun vektörlere dönüştürür
- Konum bilgisi kritiktir ("kedi köpeği kovaladı" ≠ "köpek kediyi kovaladı")
- GPT-2 öğrenilmiş konum gömmeler kullanır (eğitilebilir parametreler)
- Kosinüs benzerliği öğrenilmiş anlamsal ilişkileri ortaya çıkarır
- GPT-2 token gömmeleri çoğunlukla yazım/büyük-küçük harf varyantlarını kümeler ("King", "KING"); daha zengin anlam dikkat katmanlarından sonra ortaya çıkar

**Sırada:** Bölüm 10’da bu gömmelerle öz-dikkat (self-attention) hesaplayacak ve her token için bağlama duyarlı bir temsil oluşturacaksın.